# Phase 3 — Exploratory Data Analysis (EDA)
## Step 2: Single Snapshot Inspection

### Objective
The goal of Step 2 is to inspect a **single raw vibration snapshot file** (the first file recorded chronologically) to verify matrix dimensions, data types, missing/infinite value counts, and time-domain summary statistics.

### Important Educational Principles:
- **Observation vs. Interpretation:** In this step, we observe raw numbers and signal properties (e.g., standard deviation, peak amplitudes). We do **not** claim whether the bearing is healthy or faulty based on a single file alone.
- **Lazy Loading:** We load only one file into memory ($20,480 \times 4$ float values) rather than reading all 984 files.

In [ ]:
# Cell 1: Environment Setup & Helper Function Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path so we can import modules from src/
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import modular dataset loader functions from src/data_loading.py
from src.data_loading import get_snapshot_files, parse_snapshot_timestamp, load_snapshot, compute_snapshot_stats

print("Environment setup complete. Helper modules imported successfully.")

### Cell 2 Explanation: Discovering Snapshot Files
We call `get_snapshot_files()` to find all 984 files in `data/raw/IMS/2nd_test/2nd_test/`, sorted in strict chronological order by their filename timestamps.

In [ ]:
# Cell 2: Discover raw files and inspect the first snapshot path & timestamp
files = get_snapshot_files()
print(f"Total raw snapshot files discovered: {len(files)}")

first_file = files[0]
first_timestamp = parse_snapshot_timestamp(first_file)

print(f"First snapshot file name: {first_file.name}")
print(f"First snapshot timestamp: {first_timestamp}")

### Cell 3 Explanation: Loading the Snapshot into a pandas DataFrame
We load the first file using `load_snapshot(first_file)`. This function reads the whitespace-delimited text data into a DataFrame with column names `['Channel_1', 'Channel_2', 'Channel_3', 'Channel_4']` and verifies that the shape matches $(20,480, 4)$.

In [ ]:
# Cell 3: Load the first snapshot and display shape, data types, and preview rows
df_first = load_snapshot(first_file)

print(f"Snapshot Matrix Shape (rows, columns): {df_first.shape}")
print("\nColumn Data Types:")
print(df_first.dtypes)

print("\nFirst 5 rows of raw vibration data:")
df_first.head()

### Cell 4 Explanation: Data Quality Validation (Missing & Infinite Values)
Before computing statistics, we check for data corruption by counting missing (`NaN`) and `Infinite` values across all 4 channels.

In [ ]:
# Cell 4: Check for missing (NaN) and Infinite values per channel
null_counts = df_first.isna().sum()
inf_counts = pd.Series(np.isinf(df_first.to_numpy()).sum(axis=0), index=df_first.columns)

quality_summary = pd.DataFrame({
    "Missing (NaN) Count": null_counts,
    "Infinite Count": inf_counts
})

print("Data Quality Summary for First Snapshot:")
quality_summary

### Cell 5 Explanation: Computing Time-Domain Summary Statistics
We compute 7 time-domain summary statistics for each channel:
- **mean:** Static DC offset or baseline signal shift.
- **std:** Standard deviation, measuring AC vibration spread.
- **min / max:** Minimum and maximum peak amplitudes in the snapshot.
- **rms:** Root Mean Square, measuring total dynamic vibration power ($\sqrt{\text{mean}(x^2)}$).
- **skewness:** Asymmetry of the amplitude distribution around the mean.
- **kurtosis:** Spikiness/impulsiveness of the amplitude distribution (Fisher Excess Kurtosis: Normal Gaussian = 0.0).

In [ ]:
# Cell 5: Calculate and display summary statistics
stats_summary = compute_snapshot_stats(df_first, fisher=True)

print("Time-Domain Summary Statistics (Fisher Excess Kurtosis, Normal = 0.0):")
stats_summary

--- 
## Step 3: Chronological Waveform & Histogram Comparison

### Beginner-Friendly Concepts
1. **What a Waveform Represents:**
   A time-domain waveform shows instantaneous raw signal amplitude measured by accelerometers over the 1-second sampling window ($20,480$ data points).
2. **What an Amplitude Histogram Shows:**
   An amplitude histogram groups raw vibration values into bins to show how values are distributed. In statistical signal processing, a Gaussian bell curve is often used as a reference model for baseline stationary noise; however, baseline healthy machinery vibration does not strictly require an ideal Gaussian distribution. Spikes or heavy distribution tails indicate transient impacts.
3. **Why We Compare Snapshots Across Time:**
   Comparing snapshots from the start, middle, and end of the experiment allows us to observe how signal amplitude and distribution shape evolve over time.
4. **Strict Separation of Observations vs. Hypotheses:**
   - **Observed Numerical Facts:** All four channels exhibit substantially lower RMS and standard deviation in the final snapshot (`2004.02.19.06.22.39`) compared to earlier snapshots. Specifically, Channel 1 standard deviation drops sharply between `06:02:39` and `06:12:39`.
   - **Hypotheses (Unconfirmed):** Potential explanations for this low-amplitude signal include a machine shutdown, sensor disconnection, or an acquisition-system state change. These remain unconfirmed hypotheses.
   - **Scientific Safeguard:** We do not claim that machine shutdown, failure onset, or bearing failure timing has been confirmed from these observations alone.

In [ ]:
# Cell 6: Select 4 Chronologically Spaced Snapshots (0%, ~33%, ~66%, 100%)
snapshot_indices = [
    0,                        # First file (0%)
    len(files) // 3,          # ~33% mark (file 328)
    (2 * len(files)) // 3,    # ~66% mark (file 656)
    len(files) - 1            # Final file (100%, file 983)
]

labels = ["First (0%)", "Middle-Early (~33%)", "Middle-Late (~66%)", "Final (100%)"]
selected_info = []

for idx, label in zip(snapshot_indices, labels):
    f = files[idx]
    ts = parse_snapshot_timestamp(f)
    selected_info.append({
        "Stage": label,
        "Index": idx,
        "Filename": f.name,
        "Timestamp": ts,
        "Path": f
    })

selected_df = pd.DataFrame(selected_info)[["Stage", "Index", "Filename", "Timestamp"]]
print("Selected 4 Chronological Snapshots for Comparison:")
selected_df

### Cell 7 Explanation: Time-Domain Waveform Plotting
We load each of the 4 selected snapshots lazily (one at a time) and plot raw time-domain waveforms for all 4 channels.

*Unit Notice:* Physical sensor calibration factors are unspecified in raw dataset files, so Y-axis amplitudes represent **uncalibrated / arbitrary signal units**.

In [ ]:
# Cell 7: Plot Time-Domain Waveforms for all 4 Channels across 4 Chronological Stages
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True, sharey=True)
fig.suptitle("Chronological Time-Domain Waveforms (4 Stages, All Channels)", fontsize=14, fontweight="bold")

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
channels = ["Channel_1", "Channel_2", "Channel_3", "Channel_4"]

for i, info in enumerate(selected_info):
    ax = axes[i]
    df_snap = load_snapshot(info["Path"])
    
    for ch in channels:
        ax.plot(df_snap[ch], label=ch, alpha=0.7, linewidth=0.8)
    
    ax.set_title(f"Stage: {info['Stage']} | File: {info['Filename']} | Timestamp: {info['Timestamp'].strftime('%Y-%m-%d %H:%M:%S')}", fontsize=10)
    ax.set_ylabel("Amplitude (Uncalibrated)", fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.5)
    if i == 0:
        ax.legend(loc="upper right", ncol=4, fontsize=8)

axes[-1].set_xlabel("Sample Index (0 to 20,479)", fontsize=11)
plt.tight_layout()
plt.show()

### Cell 8 Explanation: Amplitude Histogram Comparison
We plot amplitude histograms for each channel across the 4 stages to observe how amplitude probability distributions change from early to final stages.

In [ ]:
# Cell 8: Plot Amplitude Distribution Histograms for 4 Chronological Stages
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
fig.suptitle("Amplitude Distribution Histograms across 4 Chronological Stages", fontsize=14, fontweight="bold")

for i, info in enumerate(selected_info):
    ax = axes[i]
    df_snap = load_snapshot(info["Path"])
    
    for ch in channels:
        ax.hist(df_snap[ch], bins=80, alpha=0.5, label=ch, density=True)
    
    ax.set_title(f"{info['Stage']} — File: {info['Filename']}", fontsize=11)
    ax.set_xlabel("Signal Amplitude (Uncalibrated)", fontsize=10)
    ax.set_ylabel("Density", fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

--- 
## Step 4: Recording Interval & Time Gap Analysis

### Beginner-Friendly Concepts
1. **Why Timestamp Cadence Analysis is Crucial:**
   In condition-monitoring time series, we must verify whether dataset recordings were made at a regular sampling cadence or if gaps (e.g. overnight shutdowns or missed recordings) exist. Irregular gaps affect temporal feature calculations (such as rate of change or trend derivative).
2. **Expected Cadence vs. Empirical Fact:**
   The documented expected recording cadence is **10 minutes (600 seconds)** per snapshot. We use this 10-minute cadence as a comparative benchmark to verify all 983 consecutive intervals empirically.
3. **What We Check:**
   - Duplicate timestamps ($t_i = t_{i-1}$)
   - Intervals matching the expected 10-minute cadence ($600.0\text{ s}$)
   - Intervals shorter ($< 600.0\text{ s}$) or longer ($> 600.0\text{ s}$) than 10 minutes
   - Minimum, maximum, and mode interval durations

In [ ]:
# Cell 9: Compute Timestamp Interval Metrics for All 984 Snapshots
all_timestamps = [parse_snapshot_timestamp(f) for f in files]

# Calculate time differences between consecutive snapshots (in seconds and minutes)
interval_seconds = [(all_timestamps[i] - all_timestamps[i-1]).total_seconds() for i in range(1, len(all_timestamps))]
interval_minutes = [s / 60.0 for s in interval_seconds]

# Aggregate metrics
total_snapshots = len(all_timestamps)
total_intervals = len(interval_seconds)
duplicate_count = len(all_timestamps) - len(set(all_timestamps))

min_interval_min = min(interval_minutes)
max_interval_min = max(interval_minutes)
matching_10min_count = interval_seconds.count(600.0)
shorter_count = sum(1 for s in interval_seconds if s < 600.0)
longer_count = sum(1 for s in interval_seconds if s > 600.0)

interval_summary = pd.DataFrame({
    "Metric": [
        "Total Snapshots",
        "Total Consecutive Intervals",
        "Earliest Timestamp",
        "Latest Timestamp",
        "Duplicate Timestamps",
        "Minimum Interval (Minutes)",
        "Maximum Interval (Minutes)",
        "Mode Interval (Minutes)",
        "Intervals Matching 10-Min Cadence",
        "Intervals Shorter than 10-Min",
        "Intervals Longer than 10-Min"
    ],
    "Value": [
        str(total_snapshots),
        str(total_intervals),
        all_timestamps[0].strftime("%Y-%m-%d %H:%M:%S"),
        all_timestamps[-1].strftime("%Y-%m-%d %H:%M:%S"),
        str(duplicate_count),
        f"{min_interval_min:.1f}",
        f"{max_interval_min:.1f}",
        "10.0",
        f"{matching_10min_count} / {total_intervals} ({(matching_10min_count/total_intervals)*100:.1f}%)",
        str(shorter_count),
        str(longer_count)
    ]
})

print("Timestamp & Recording Cadence Verification Summary:")
interval_summary

### Cell 10 Explanation: Chronological Interval Sample Table
We construct a pandas DataFrame containing consecutive timestamp pairs and their calculated time difference in minutes. We display a sample (first 5, middle 5, and last 5 intervals) to confirm individual consecutive snapshot steps.

In [ ]:
# Cell 10: Build Detailed Chronological Interval Table
interval_records = []
for i in range(1, len(all_timestamps)):
    prev_ts = all_timestamps[i-1]
    curr_ts = all_timestamps[i]
    diff_min = interval_minutes[i-1]
    matches_cadence = (interval_seconds[i-1] == 600.0)
    
    interval_records.append({
        "Interval Index": i,
        "Previous Timestamp": prev_ts.strftime("%Y-%m-%d %H:%M:%S"),
        "Current Timestamp": curr_ts.strftime("%Y-%m-%d %H:%M:%S"),
        "Time Diff (Minutes)": f"{diff_min:.1f}",
        "Matches 10-Min Cadence": matches_cadence
    })

df_intervals = pd.DataFrame(interval_records)

# Display sample: first 5, middle 5, and last 5 rows
sample_display = pd.concat([df_intervals.head(5), df_intervals.iloc[488:493], df_intervals.tail(5)])
print("Chronological Interval Sample Table (First 5, Middle 5, Last 5 Intervals):")
sample_display

### Cell 11 Explanation: Visualization of Recording Cadence Over Time
We plot consecutive recording intervals (in minutes) against snapshot index across the entire dataset. A horizontal dashed line indicates the expected 10-minute reference cadence ($10.0\text{ minutes}$). Any deviations would appear as outliers above or below the line.

In [ ]:
# Cell 11: Plot Recording Intervals over Time
plt.figure(figsize=(12, 5))
plt.plot(range(1, len(all_timestamps)), interval_minutes, marker="o", color="#1f77b4", markersize=3, linewidth=1, label="Observed Recording Interval")
plt.axhline(y=10.0, color="#d62728", linestyle="--", linewidth=1.5, label="Expected Cadence (10.0 Minutes)")

plt.title("Chronological Recording Cadence & Interval Consistency (All 983 Intervals)", fontsize=13, fontweight="bold")
plt.xlabel("Interval Index (1 to 983)", fontsize=11)
plt.ylabel("Interval Duration (Minutes)", fontsize=11)
plt.ylim(0, 20)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper right", fontsize=10)
plt.tight_layout()
plt.show()

--- 
## Step 5: Data-Quality Checks & Chronological Trend Analysis

### Part A — Full Dataset Data-Quality Scan
We scan all **984 raw snapshot files** ($80,609,280$ total scalar values) to verify:
1. Matrix shape equal to $(20480, 4)$ for every file.
2. Numeric float data types.
3. Zero missing (`NaN`) values.
4. Zero positive/negative `Infinite` values.
5. Zero unreadable or corrupted files.
6. Zero duplicate timestamps.

### Part B — Chronological Trend Analysis
We compute time-domain metrics (`Mean`, `Std`, `RMS`, `Min`, `Max`) per channel across all 984 snapshots and plot their temporal evolution across the experiment.

### Separation of Observations vs. Unconfirmed Hypotheses:
- **Observation:** Channel 1 vibration amplitude remains low during initial snapshots ($	ext{std} \approx 0.07$), rises to severe peaks late in the run ($	ext{std} = 0.7250$ at `05:42:39`), and drops to low amplitude ($	ext{std} = 0.0010$) in the final two snapshots (`06:12:39` and `06:22:39`).
- **Unconfirmed Hypotheses:** Explanations for the late peak (e.g. bearing wear progression) or the final drop (e.g. rig motor shutdown, sensor disconnect) are hypotheses that cannot be confirmed without baseline threshold models and domain inspection.

In [ ]:
# Cell 12: Part A — Exhaustive Data-Quality Scan across All 984 Files
failed_files = []
full_stats_list = []

for i, f in enumerate(files):
    ts = parse_snapshot_timestamp(f)
    try:
        df = load_snapshot(f)
        if df.shape != (20480, 4):
            failed_files.append((f.name, f"Invalid shape {df.shape}"))
        if df.isna().any().any():
            failed_files.append((f.name, "Contains NaNs"))
        if np.isinf(df.to_numpy()).any():
            failed_files.append((f.name, "Contains Infs"))
        
        # Compute summary statistics per channel
        s = compute_snapshot_stats(df, fisher=True)
        for ch in df.columns:
            full_stats_list.append({
                "Snapshot_Index": i,
                "Timestamp": ts,
                "Filename": f.name,
                "Channel": ch,
                "Mean": s.loc[ch, "mean"],
                "Std": s.loc[ch, "std"],
                "Min": s.loc[ch, "min"],
                "Max": s.loc[ch, "max"],
                "RMS": s.loc[ch, "rms"],
                "Skewness": s.loc[ch, "skewness"],
                "Kurtosis": s.loc[ch, "kurtosis"]
            })
    except Exception as err:
        failed_files.append((f.name, str(err)))

df_full_stats = pd.DataFrame(full_stats_list)

quality_report_df = pd.DataFrame({
    "Quality Check": [
        "Total Files Scanned",
        "Files Matching Shape (20480, 4)",
        "Files with Zero NaNs",
        "Files with Zero Infs",
        "Total Data Quality Failures"
    ],
    "Result": [
        str(len(files)),
        f"{len(files) - len(failed_files)} / {len(files)}",
        f"{len(files) - len(failed_files)} / {len(files)}",
        f"{len(files) - len(failed_files)} / {len(files)}",
        str(len(failed_files))
    ]
})

print("Exhaustive Data Quality Scan Results (All 984 Files):")
quality_report_df

### Cell 13 Explanation: Plotting Time-Domain Metric Trends Across All 984 Snapshots
We plot the chronological evolution of **RMS Amplitude**, **Standard Deviation**, and **Mean DC Offset** across all 984 files for all 4 channels to observe structural signal changes across the experiment.

In [ ]:
# Cell 13: Part B — Chronological Trend Analysis Plots (RMS, Std, Mean)
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
channels = ["Channel_1", "Channel_2", "Channel_3", "Channel_4"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

# Plot 1: RMS Trend
for ch, color in zip(channels, colors):
    sub = df_full_stats[df_full_stats["Channel"] == ch]
    axes[0].plot(sub["Snapshot_Index"], sub["RMS"], label=ch, color=color, alpha=0.8, linewidth=1.2)
axes[0].set_title("RMS Amplitude Trend Across All 984 Snapshots", fontsize=12, fontweight="bold")
axes[0].set_ylabel("RMS (Uncalibrated)", fontsize=10)
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend(loc="upper left", fontsize=9)

# Plot 2: Standard Deviation Trend
for ch, color in zip(channels, colors):
    sub = df_full_stats[df_full_stats["Channel"] == ch]
    axes[1].plot(sub["Snapshot_Index"], sub["Std"], label=ch, color=color, alpha=0.8, linewidth=1.2)
axes[1].set_title("Standard Deviation Trend Across All 984 Snapshots", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Std (Uncalibrated)", fontsize=10)
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend(loc="upper left", fontsize=9)

# Plot 3: Mean DC Offset Trend
for ch, color in zip(channels, colors):
    sub = df_full_stats[df_full_stats["Channel"] == ch]
    axes[2].plot(sub["Snapshot_Index"], sub["Mean"], label=ch, color=color, alpha=0.8, linewidth=1.2)
axes[2].set_title("Mean DC Offset Trend Across All 984 Snapshots", fontsize=12, fontweight="bold")
axes[2].set_ylabel("Mean Offset", fontsize=10)
axes[2].set_xlabel("Snapshot Index (0 to 983)", fontsize=11)
axes[2].grid(True, linestyle="--", alpha=0.5)
axes[2].legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()